In [ ]:
import os
import glob
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid, save_image

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
class Config:
    # Dataset paths
    BASE_DIR    = "/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs"
    PHOTO_DIR   = os.path.join(BASE_DIR, "photos")      # Real face photos
    SKETCH_DIR  = os.path.join(BASE_DIR, "sketches")    # Hand-drawn sketches
    OUTPUT_DIR  = "/kaggle/working/outputs"
    CKPT_DIR    = "/kaggle/working/checkpoints"

    # Image settings
    IMG_SIZE    = 256        # Resize to 256×256
    CHANNELS    = 3          # RGB

    # Training hyperparameters
    BATCH_SIZE  = 16
    NUM_EPOCHS  = 100
    LR          = 2e-4
    BETAS       = (0.5, 0.999)
    LAMBDA_L1   = 100        # Weight for L1 loss

    # Data split
    TRAIN_RATIO = 0.8
    VAL_RATIO   = 0.1

    # Training utilities
    SAVE_FREQ   = 5          # Save checkpoint every N epochs
    VIZ_FREQ    = 10         # Visualize every N epochs
    NUM_WORKERS = 2
    USE_AMP     = True       # Mixed precision
    MULTI_GPU   = True       # DataParallel if >1 GPU

    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()

# Create output directories
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
os.makedirs(cfg.CKPT_DIR,   exist_ok=True)
os.makedirs(os.path.join(cfg.OUTPUT_DIR, "samples"), exist_ok=True)

print(f"Device: {cfg.DEVICE}")
print(f"Image Size: {cfg.IMG_SIZE}×{cfg.IMG_SIZE}")
print(f"Batch Size: {cfg.BATCH_SIZE}")
print(f"Epochs: {cfg.NUM_EPOCHS}")

In [ ]:
def explore_dataset(cfg):
    """Explore dataset structure and find paired images."""
    print("=" * 60)
    print("DATASET EXPLORATION")
    print("=" * 60)

    # List all directories in BASE_DIR
    for d in sorted(os.listdir(cfg.BASE_DIR)):
        full_path = os.path.join(cfg.BASE_DIR, d)
        if os.path.isdir(full_path):
            files = os.listdir(full_path)
            exts  = set(os.path.splitext(f)[1].lower() for f in files)
            print(f"  {d}/  →  {len(files)} files  |  exts: {exts}")

    # Check primary photo & sketch directories
    photo_exts   = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    sketch_exts  = ['*.jpg', '*.jpeg', '*.png', '*.bmp']

    photo_files  = []
    sketch_files = []

    for ext in photo_exts:
        photo_files  += glob.glob(os.path.join(cfg.PHOTO_DIR,  ext))
        sketch_files += glob.glob(os.path.join(cfg.SKETCH_DIR, ext))

    # If primary dirs empty, try alternatives
    if len(photo_files) == 0:
        for alt in ["photo", "photos", "cropped_sketch"]:
            p = os.path.join(cfg.BASE_DIR, alt)
            if os.path.exists(p):
                for ext in photo_exts:
                    photo_files += glob.glob(os.path.join(p, ext))
                if photo_files:
                    cfg.PHOTO_DIR = p
                    print(f"\nUsing alternative photo dir: {alt}/")
                    break

    if len(sketch_files) == 0:
        for alt in ["sketch", "sketches", "original_sketch"]:
            p = os.path.join(cfg.BASE_DIR, alt)
            if os.path.exists(p):
                for ext in sketch_exts:
                    sketch_files += glob.glob(os.path.join(p, ext))
                if sketch_files:
                    cfg.SKETCH_DIR = p
                    print(f"Using alternative sketch dir: {alt}/")
                    break

    print(f"\nPhotos  found: {len(photo_files)}  (in {cfg.PHOTO_DIR})")
    print(f"Sketches found: {len(sketch_files)}  (in {cfg.SKETCH_DIR})")

    if photo_files:
        print(f"  Sample photo: {os.path.basename(photo_files[0])}")
    if sketch_files:
        print(f"  Sample sketch: {os.path.basename(sketch_files[0])}")

    return photo_files, sketch_files

photo_files, sketch_files = explore_dataset(cfg)

In [ ]:
import os
from pathlib import Path

def find_paired_images(photo_files, sketch_files):
    """
    Match sketch and photo files by stem name.
    CUHK naming: f-001-01.jpg (photo) <-> f-001-01.jpg (sketch)
    """
    photo_map = {Path(f).stem: f for f in photo_files}
    sketch_map = {Path(f).stem: f for f in sketch_files}
    
    print(f"Total photo stems: {len(photo_map)}")
    print(f"Total sketch stems: {len(sketch_map)}")
    
    # Direct name match
    common = set(photo_map.keys()) & set(sketch_map.keys())
    print(f"Direct matches found: {len(common)}")
    
    # If no direct matches, try fuzzy matching
    if len(common) == 0:
        print("No exact matches found, trying fuzzy matching...")
        pairs_temp = []
        
        for photo_stem, photo_path in photo_map.items():
            for sketch_stem, sketch_path in sketch_map.items():
                # Check various matching patterns
                if (photo_stem in sketch_stem or 
                    sketch_stem in photo_stem or
                    photo_stem.replace('photo', '') == sketch_stem.replace('sketch', '') or
                    photo_stem.split('_')[-1] == sketch_stem.split('_')[-1] or
                    photo_stem == sketch_stem.split('_')[0] or
                    sketch_stem == photo_stem.split('_')[0]):
                    
                    pairs_temp.append((photo_path, sketch_path))
                    break  # Match found for this photo
        
        print(f"Found {len(pairs_temp)} pairs through fuzzy matching")
        return pairs_temp
    
    # Build pairs only from keys that exist in BOTH dictionaries
    pairs = []
    for k in sorted(common):
        if k in photo_map and k in sketch_map:
            pairs.append((photo_map[k], sketch_map[k]))
    
    print(f"Paired images found: {len(pairs)}")
    return pairs

# Your file lists (replace with your actual file loading code)
# Example:
# photo_files = [f for f in os.listdir('photos') if f.endswith('.jpg')]
# sketch_files = [f for f in os.listdir('sketches') if f.endswith('.jpg')]

pairs = find_paired_images(photo_files, sketch_files)

# If pairing fails, zip sorted lists (CUHK is inherently paired by index)
if len(pairs) == 0:
    print("Falling back to sorted zip pairing...")
    n = min(len(photo_files), len(sketch_files))
    photo_s = sorted(photo_files)
    sketch_s = sorted(sketch_files)
    pairs = list(zip(photo_s[:n], sketch_s[:n]))
    print(f"  Paired by sort order: {len(pairs)} pairs")

print(f"\nExample pair:")
if pairs:
    print(f"  Photo:  {os.path.basename(pairs[0][0])}")
    print(f"  Sketch: {os.path.basename(pairs[0][1])}")
    
# Optional: Print first 10 pairs to verify
print(f"\nFirst 10 pairs:")
for i, (photo, sketch) in enumerate(pairs[:10]):
    print(f"{i+1}. Photo: {os.path.basename(photo)} <-> Sketch: {os.path.basename(sketch)}")

In [ ]:
# Visualize some raw paired images
def show_raw_pairs(pairs, n=5):
    n = min(n, len(pairs))
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    fig.suptitle("CUHK Dataset — Raw Paired Samples", fontsize=14, fontweight='bold')

    for i, (photo_path, sketch_path) in enumerate(random.sample(pairs, n)):
        photo  = Image.open(photo_path).convert('RGB')
        sketch = Image.open(sketch_path).convert('RGB')

        axes[0, i].imshow(photo)
        axes[0, i].set_title(f"Photo\n{os.path.basename(photo_path)}", fontsize=8)
        axes[0, i].axis('off')

        axes[1, i].imshow(sketch)
        axes[1, i].set_title(f"Sketch\n{os.path.basename(sketch_path)}", fontsize=8)
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.OUTPUT_DIR, "raw_pairs.png"), dpi=100, bbox_inches='tight')
    plt.show()

show_raw_pairs(pairs, n=5)

In [ ]:
class CUHKSketchDataset(Dataset):
    """
    CUHK Face Sketch Dataset.
    Returns: (sketch_tensor, photo_tensor) both in [-1, 1]
    """
    def __init__(self, pairs, img_size=256, augment=False):
        self.pairs    = pairs
        self.img_size = img_size
        self.augment  = augment

        # Base transforms applied to both
        self.resize = transforms.Resize((img_size, img_size), 
                                         interpolation=transforms.InterpolationMode.BICUBIC)
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize([0.5]*3, [0.5]*3)  # → [-1, 1]

    def __len__(self):
        return len(self.pairs)

    def apply_transform(self, pil_img):
        img = self.resize(pil_img)
        img = self.to_tensor(img)
        img = self.normalize(img)
        return img

    def apply_augment(self, sketch_pil, photo_pil):
        """Consistent augmentation for both input and target."""
        # Random horizontal flip
        if random.random() > 0.5:
            sketch_pil = transforms.functional.hflip(sketch_pil)
            photo_pil  = transforms.functional.hflip(photo_pil)

        # Random crop (resize to 286 then crop to 256)
        resize286 = transforms.Resize((286, 286), 
                                       interpolation=transforms.InterpolationMode.BICUBIC)
        sketch_pil = resize286(sketch_pil)
        photo_pil  = resize286(photo_pil)

        i, j, h, w = transforms.RandomCrop.get_params(
            sketch_pil, output_size=(self.img_size, self.img_size))
        sketch_pil = transforms.functional.crop(sketch_pil, i, j, h, w)
        photo_pil  = transforms.functional.crop(photo_pil,  i, j, h, w)

        return sketch_pil, photo_pil

    def __getitem__(self, idx):
        photo_path, sketch_path = self.pairs[idx]

        photo_pil  = Image.open(photo_path).convert('RGB')
        sketch_pil = Image.open(sketch_path).convert('RGB')

        if self.augment:
            sketch_pil, photo_pil = self.apply_augment(sketch_pil, photo_pil)

        sketch = self.apply_transform(sketch_pil)
        photo  = self.apply_transform(photo_pil)

        # Input: sketch (condition), Target: photo (real)
        return sketch, photo


# Split data
random.shuffle(pairs)
n_total = len(pairs)
n_train = int(n_total * cfg.TRAIN_RATIO)
n_val   = int(n_total * cfg.VAL_RATIO)
n_test  = n_total - n_train - n_val

train_pairs = pairs[:n_train]
val_pairs   = pairs[n_train:n_train+n_val]
test_pairs  = pairs[n_train+n_val:]

print(f"Total pairs : {n_total}")
print(f"Train pairs : {len(train_pairs)}")
print(f"Val pairs   : {len(val_pairs)}")
print(f"Test pairs  : {len(test_pairs)}")

# Create Datasets
train_dataset = CUHKSketchDataset(train_pairs, img_size=cfg.IMG_SIZE, augment=True)
val_dataset   = CUHKSketchDataset(val_pairs,   img_size=cfg.IMG_SIZE, augment=False)
test_dataset  = CUHKSketchDataset(test_pairs,  img_size=cfg.IMG_SIZE, augment=False)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")

In [ ]:
# Verify dataset output
sketch_sample, photo_sample = next(iter(train_loader))
print(f"Sketch batch shape : {sketch_sample.shape}")
print(f"Photo batch shape  : {photo_sample.shape}")
print(f"Sketch value range : [{sketch_sample.min():.2f}, {sketch_sample.max():.2f}]")
print(f"Photo value range  : [{photo_sample.min():.2f}, {photo_sample.max():.2f}]")# Verify dataset output
sketch_sample, photo_sample = next(iter(train_loader))
print(f"Sketch batch shape : {sketch_sample.shape}")
print(f"Photo batch shape  : {photo_sample.shape}")
print(f"Sketch value range : [{sketch_sample.min():.2f}, {sketch_sample.max():.2f}]")
print(f"Photo value range  : [{photo_sample.min():.2f}, {photo_sample.max():.2f}]")

In [ ]:
class UNetBlock(nn.Module):
    """
    U-Net block used in both encoder and decoder.
    
    Encoder block: Conv → BatchNorm → LeakyReLU
    Decoder block: ConvTranspose → BatchNorm → (Dropout) → ReLU
    """
    def __init__(self, in_ch, out_ch, down=True, bn=True, dropout=False, act=True):
        super().__init__()
        self.act     = act
        self.dropout = dropout
        self.down    = down

        if down:
            # Encoder: stride-2 conv
            self.conv = nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=not bn)
        else:
            # Decoder: transposed conv (upsample)
            self.conv = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=not bn)

        self.bn  = nn.BatchNorm2d(out_ch) if bn else nn.Identity()
        self.drop = nn.Dropout(0.5) if dropout else nn.Identity()
        self.relu = nn.ReLU(inplace=True) if not down else nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        if self.dropout:
            x = self.drop(x)
        if self.act:
            x = self.relu(x)
        return x


class UNetGenerator(nn.Module):
    """
    U-Net Generator for Pix2Pix.

    Architecture (256×256 input):
      Encoder: 8 down-sampling blocks
      Decoder: 8 up-sampling blocks with skip connections

    Input : (B, 3, 256, 256)  — sketch/edge image
    Output: (B, 3, 256, 256)  — realistic face image
    """
    def __init__(self, in_ch=3, out_ch=3, ngf=64):
        super().__init__()

        # ── Encoder (downsampling) ──────────────────────────────────────────
        # e1: 256→128,  e2: 128→64,  e3: 64→32,  e4: 32→16
        # e5:  16→8,   e6:   8→4,   e7:  4→2,   e8:  2→1
        self.e1 = UNetBlock(in_ch,      ngf,    down=True, bn=False)  # no BN first layer
        self.e2 = UNetBlock(ngf,        ngf*2,  down=True)
        self.e3 = UNetBlock(ngf*2,      ngf*4,  down=True)
        self.e4 = UNetBlock(ngf*4,      ngf*8,  down=True)
        self.e5 = UNetBlock(ngf*8,      ngf*8,  down=True)
        self.e6 = UNetBlock(ngf*8,      ngf*8,  down=True)
        self.e7 = UNetBlock(ngf*8,      ngf*8,  down=True)
        self.e8 = UNetBlock(ngf*8,      ngf*8,  down=True, bn=False)  # bottleneck

        # ── Decoder (upsampling) with skip connections ──────────────────────
        # Skip: concat encoder output → decoder input channels double
        # d1:  1→2,   d2:  2→4,   d3:  4→8,   d4:  8→16
        # d5: 16→32, d6: 32→64,  d7: 64→128, d8: 128→256
        self.d1 = UNetBlock(ngf*8,      ngf*8,  down=False, dropout=True)
        self.d2 = UNetBlock(ngf*8*2,    ngf*8,  down=False, dropout=True)
        self.d3 = UNetBlock(ngf*8*2,    ngf*8,  down=False, dropout=True)
        self.d4 = UNetBlock(ngf*8*2,    ngf*8,  down=False)
        self.d5 = UNetBlock(ngf*8*2,    ngf*4,  down=False)
        self.d6 = UNetBlock(ngf*4*2,    ngf*2,  down=False)
        self.d7 = UNetBlock(ngf*2*2,    ngf,    down=False)

        # Final output layer
        self.final = nn.Sequential(
            nn.ConvTranspose2d(ngf*2, out_ch, 4, stride=2, padding=1),
            nn.Tanh()   # Output in [-1, 1]
        )

    def forward(self, x):
        # Encoder — save skip features
        e1 = self.e1(x)   # (B, 64,  128, 128)
        e2 = self.e2(e1)  # (B, 128,  64,  64)
        e3 = self.e3(e2)  # (B, 256,  32,  32)
        e4 = self.e4(e3)  # (B, 512,  16,  16)
        e5 = self.e5(e4)  # (B, 512,   8,   8)
        e6 = self.e6(e5)  # (B, 512,   4,   4)
        e7 = self.e7(e6)  # (B, 512,   2,   2)
        e8 = self.e8(e7)  # (B, 512,   1,   1)  — bottleneck

        # Decoder — concat skip connections (U-Net style)
        d1 = self.d1(e8)                     # (B, 512,  2,  2)
        d2 = self.d2(torch.cat([d1, e7], 1)) # (B, 512,  4,  4)
        d3 = self.d3(torch.cat([d2, e6], 1)) # (B, 512,  8,  8)
        d4 = self.d4(torch.cat([d3, e5], 1)) # (B, 512, 16, 16)
        d5 = self.d5(torch.cat([d4, e4], 1)) # (B, 256, 32, 32)
        d6 = self.d6(torch.cat([d5, e3], 1)) # (B, 128, 64, 64)
        d7 = self.d7(torch.cat([d6, e2], 1)) # (B,  64,128,128)
        out = self.final(torch.cat([d7, e1], 1)) # (B, 3, 256,256)

        return out


# Test generator
G = UNetGenerator(in_ch=3, out_ch=3, ngf=64)
dummy = torch.randn(1, 3, 256, 256)
out   = G(dummy)
print(f"Generator output shape: {out.shape}")

total_params = sum(p.numel() for p in G.parameters())
trainable    = sum(p.numel() for p in G.parameters() if p.requires_grad)
print(f"Generator parameters: {total_params:,} total  |  {trainable:,} trainable")

In [ ]:
class PatchGANDiscriminator(nn.Module):
    """
    PatchGAN Discriminator for Pix2Pix.

    Classifies overlapping 70×70 patches as real or fake.
    Input : (B, 6, H, W) — concatenation of sketch + photo/generated
    Output: (B, 1, h, w) — patch-level probability map

    Note: The paper's "PatchGAN" with 70×70 receptive field uses
    4 layers of stride-2 convolutions.
    """
    def __init__(self, in_ch=6, ndf=64):
        super().__init__()

        def conv_block(in_c, out_c, stride=2, bn=True):
            layers = [nn.Conv2d(in_c, out_c, 4, stride=stride, padding=1, bias=not bn)]
            if bn:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        self.model = nn.Sequential(
            # Layer 1: no BN  (256→128)
            conv_block(in_ch,    ndf,   stride=2, bn=False),
            # Layer 2         (128→64)
            conv_block(ndf,      ndf*2, stride=2),
            # Layer 3         (64→32)
            conv_block(ndf*2,    ndf*4, stride=2),
            # Layer 4         (32→31 with stride=1)
            conv_block(ndf*4,    ndf*8, stride=1),
            # Output layer    (31→30)
            nn.Conv2d(ndf*8, 1, 4, stride=1, padding=1)
            # No sigmoid — use BCEWithLogitsLoss
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, sketch, image):
        """Concatenate condition (sketch) and image along channel dim."""
        x = torch.cat([sketch, image], dim=1)  # (B, 6, H, W)
        return self.model(x)


# Test discriminator
D = PatchGANDiscriminator(in_ch=6, ndf=64)
dummy_s = torch.randn(1, 3, 256, 256)
dummy_p = torch.randn(1, 3, 256, 256)
out_d   = D(dummy_s, dummy_p)
print(f"Discriminator output shape: {out_d.shape}")

total_params_d = sum(p.numel() for p in D.parameters())
print(f"Discriminator parameters: {total_params_d:,}")

print("\nPatchGAN receptive field: ~70×70 pixels")

In [ ]:
class Pix2PixLoss:
    """
    Pix2Pix loss functions.

    Total Generator Loss   = GAN_Loss + λ * L1_Loss
    Total Discriminator Loss = GAN_Loss(real) + GAN_Loss(fake)
    """
    def __init__(self, lambda_l1=100, device='cpu'):
        self.lambda_l1 = lambda_l1
        self.device    = device
        self.bce_loss  = nn.BCEWithLogitsLoss()
        self.l1_loss   = nn.L1Loss()

    def _target(self, pred, is_real):
        """Create label tensors matching discriminator output shape."""
        val = 1.0 if is_real else 0.0
        return torch.full_like(pred, val, device=self.device)

    # ── Discriminator losses ────────────────────────────────────────────────
    def d_real_loss(self, pred_real):
        """D should predict 1 for real pairs."""
        return self.bce_loss(pred_real, self._target(pred_real, True))

    def d_fake_loss(self, pred_fake):
        """D should predict 0 for fake pairs."""
        return self.bce_loss(pred_fake, self._target(pred_fake, False))

    def discriminator_loss(self, pred_real, pred_fake):
        """Total discriminator loss = (real_loss + fake_loss) / 2"""
        return (self.d_real_loss(pred_real) + self.d_fake_loss(pred_fake)) * 0.5

    # ── Generator losses ────────────────────────────────────────────────────
    def g_adv_loss(self, pred_fake):
        """G wants D to predict 1 for generated images."""
        return self.bce_loss(pred_fake, self._target(pred_fake, True))

    def g_l1_loss(self, generated, real):
        """Pixel-wise reconstruction loss."""
        return self.l1_loss(generated, real)

    def generator_loss(self, pred_fake, generated, real):
        """Total generator loss = adversarial + λ·L1"""
        adv = self.g_adv_loss(pred_fake)
        l1  = self.g_l1_loss(generated, real)
        total = adv + self.lambda_l1 * l1
        return total, adv, l1

print("Loss functions defined.")
print(f"  Adversarial: BCEWithLogitsLoss")
print(f"  Reconstruction: L1Loss  (λ = {cfg.LAMBDA_L1})")
print(f"  Total G Loss = G_adv + {cfg.LAMBDA_L1} × L1")

In [ ]:
def build_models(cfg):
    """Initialize Generator and Discriminator, move to device, wrap DataParallel."""
    G = UNetGenerator(in_ch=3, out_ch=3, ngf=64)
    D = PatchGANDiscriminator(in_ch=6, ndf=64)

    # Weight initialization
    def init_weights(m):
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.zeros_(m.bias)

    G.apply(init_weights)
    D.apply(init_weights)

    G = G.to(cfg.DEVICE)
    D = D.to(cfg.DEVICE)

    # Multi-GPU wrapping
    if cfg.MULTI_GPU and torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        G = nn.DataParallel(G)
        D = nn.DataParallel(D)

    return G, D


G, D = build_models(cfg)

# Optimizers
opt_G = optim.Adam(G.parameters(), lr=cfg.LR, betas=cfg.BETAS)
opt_D = optim.Adam(D.parameters(), lr=cfg.LR, betas=cfg.BETAS)

# Linear LR decay (after epoch 50, decay to 0 by epoch 100)
def lr_lambda(epoch):
    decay_start = cfg.NUM_EPOCHS // 2
    if epoch < decay_start:
        return 1.0
    return 1.0 - (epoch - decay_start) / (cfg.NUM_EPOCHS - decay_start)

sched_G = optim.lr_scheduler.LambdaLR(opt_G, lr_lambda=lr_lambda)
sched_D = optim.lr_scheduler.LambdaLR(opt_D, lr_lambda=lr_lambda)

# Loss function
criterion = Pix2PixLoss(lambda_l1=cfg.LAMBDA_L1, device=cfg.DEVICE)

# Mixed precision scaler
scaler_G = GradScaler(enabled=cfg.USE_AMP)
scaler_D = GradScaler(enabled=cfg.USE_AMP)

print("\nModel architecture ready.")
print(f"Optimizer: Adam  LR={cfg.LR}  Betas={cfg.BETAS}")
print(f"LR Scheduler: Linear decay from epoch {cfg.NUM_EPOCHS//2}")
print(f"Mixed Precision: {cfg.USE_AMP}")

In [ ]:
def save_checkpoint(epoch, G, D, opt_G, opt_D, logs, path):
    ckpt = {
        'epoch': epoch,
        'G_state': G.state_dict(),
        'D_state': D.state_dict(),
        'opt_G':   opt_G.state_dict(),
        'opt_D':   opt_D.state_dict(),
        'logs':    logs,
    }
    torch.save(ckpt, path)
    print(f"  ✓ Checkpoint saved: {os.path.basename(path)}")


def load_checkpoint(path, G, D, opt_G, opt_D):
    ckpt = torch.load(path, map_location=cfg.DEVICE)
    G.load_state_dict(ckpt['G_state'])
    D.load_state_dict(ckpt['D_state'])
    opt_G.load_state_dict(ckpt['opt_G'])
    opt_D.load_state_dict(ckpt['opt_D'])
    return ckpt['epoch'], ckpt['logs']


@torch.no_grad()
def validate(G, val_loader, criterion, device):
    """Compute validation L1 loss."""
    G.eval()
    total_l1 = 0.0
    for sketch, real in val_loader:
        sketch = sketch.to(device)
        real   = real.to(device)
        fake   = G(sketch)
        total_l1 += criterion.l1_loss(fake, real).item()
    G.train()
    return total_l1 / len(val_loader)


def visualize_samples(G, dataset, device, n=8, epoch=0, save_dir=None):
    """Show sketch | generated | real side by side."""
    G.eval()
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    sketches, reals, fakes = [], [], []
    with torch.no_grad():
        for idx in indices:
            sketch, real = dataset[idx]
            sketch = sketch.unsqueeze(0).to(device)
            fake   = G(sketch)
            sketches.append(sketch.cpu())
            reals.append(real.unsqueeze(0))
            fakes.append(fake.cpu())

    def denorm(t):
        return (t * 0.5 + 0.5).clamp(0, 1)

    cols = len(indices)
    fig, axes = plt.subplots(3, cols, figsize=(3*cols, 9))
    fig.suptitle(f"Epoch {epoch} — Sketch | Generated | Ground Truth",
                 fontsize=13, fontweight='bold')

    row_labels = ['Input\nSketch', 'Generated\nPhoto', 'Ground\nTruth']
    for r, (row_data, label) in enumerate(zip([sketches, fakes, reals], row_labels)):
        for c, img in enumerate(row_data):
            axes[r, c].imshow(denorm(img[0]).permute(1, 2, 0).numpy())
            axes[r, c].axis('off')
            if c == 0:
                axes[r, c].set_ylabel(label, fontsize=10, rotation=0,
                                       labelpad=60, va='center')

    plt.tight_layout()
    if save_dir:
        path = os.path.join(save_dir, f"samples_epoch_{epoch:04d}.png")
        plt.savefig(path, dpi=100, bbox_inches='tight')
    plt.show()
    G.train()

print("Training utilities defined.")

In [ ]:
# Training history
logs = {
    'g_loss': [], 'd_loss': [], 'g_adv': [], 'g_l1': [], 'val_l1': []
}

# ──────────────────────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP
# ──────────────────────────────────────────────────────────────────────────────
print("Starting Pix2Pix training...")
print(f"  Epochs   : {cfg.NUM_EPOCHS}")
print(f"  Batch    : {cfg.BATCH_SIZE}")
print(f"  AMP      : {cfg.USE_AMP}")
print("=" * 70)

start_time = time.time()

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    G.train()
    D.train()

    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    epoch_g_adv  = 0.0
    epoch_g_l1   = 0.0

    for batch_idx, (sketch, real) in enumerate(train_loader):
        sketch = sketch.to(cfg.DEVICE)
        real   = real.to(cfg.DEVICE)

        # ── 1. Train Discriminator ──────────────────────────────────────────
        opt_D.zero_grad(set_to_none=True)

        with autocast(enabled=cfg.USE_AMP):
            # Generate fake images (no grad needed for G here)
            fake = G(sketch).detach()

            # D(sketch, real_photo) — should be 1
            pred_real = D(sketch, real)
            # D(sketch, fake_photo) — should be 0
            pred_fake = D(sketch, fake)

            loss_D = criterion.discriminator_loss(pred_real, pred_fake)

        scaler_D.scale(loss_D).backward()
        scaler_D.step(opt_D)
        scaler_D.update()

        # ── 2. Train Generator ──────────────────────────────────────────────
        opt_G.zero_grad(set_to_none=True)

        with autocast(enabled=cfg.USE_AMP):
            fake     = G(sketch)
            pred_fake_for_G = D(sketch, fake)

            loss_G, loss_adv, loss_l1 = criterion.generator_loss(
                pred_fake_for_G, fake, real)

        scaler_G.scale(loss_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()

        # Accumulate losses
        epoch_g_loss += loss_G.item()
        epoch_d_loss += loss_D.item()
        epoch_g_adv  += loss_adv.item()
        epoch_g_l1   += loss_l1.item()

    # ── Per-epoch averages ──────────────────────────────────────────────────
    n_batches = len(train_loader)
    avg_g = epoch_g_loss / n_batches
    avg_d = epoch_d_loss / n_batches
    avg_adv = epoch_g_adv / n_batches
    avg_l1  = epoch_g_l1  / n_batches

    # Validation
    val_l1 = validate(G, val_loader, criterion, cfg.DEVICE)

    logs['g_loss'].append(avg_g)
    logs['d_loss'].append(avg_d)
    logs['g_adv'].append(avg_adv)
    logs['g_l1'].append(avg_l1)
    logs['val_l1'].append(val_l1)

    # LR step
    sched_G.step()
    sched_D.step()

    elapsed = time.time() - start_time
    print(f"Epoch [{epoch:3d}/{cfg.NUM_EPOCHS}]  "
          f"G: {avg_g:.4f}  D: {avg_d:.4f}  "
          f"Adv: {avg_adv:.4f}  L1: {avg_l1:.4f}  "
          f"Val-L1: {val_l1:.4f}  "
          f"LR_G: {sched_G.get_last_lr()[0]:.6f}  "
          f"[{elapsed/60:.1f}min]")

    # ── Checkpoint ─────────────────────────────────────────────────────────
    if epoch % cfg.SAVE_FREQ == 0 or epoch == cfg.NUM_EPOCHS:
        ckpt_path = os.path.join(cfg.CKPT_DIR, f"pix2pix_epoch_{epoch:04d}.pth")
        save_checkpoint(epoch, G, D, opt_G, opt_D, logs, ckpt_path)

    # ── Visualization ───────────────────────────────────────────────────────
    if epoch % cfg.VIZ_FREQ == 0 or epoch == 1:
        visualize_samples(G, val_dataset, cfg.DEVICE,
                          n=8, epoch=epoch,
                          save_dir=os.path.join(cfg.OUTPUT_DIR, "samples"))

print("\nTraining complete!")
print(f"Total time: {(time.time()-start_time)/60:.1f} minutes")

In [ ]:
def plot_training_logs(logs, save_path=None):
    epochs = range(1, len(logs['g_loss']) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Pix2Pix Training Logs — CUHK Face Sketch",
                 fontsize=15, fontweight='bold')

    # 1. Generator Total Loss
    ax = axes[0, 0]
    ax.plot(epochs, logs['g_loss'], 'b-', linewidth=1.5, label='Train G Loss')
    ax.set_title('Generator Loss vs Epochs', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Discriminator Loss
    ax = axes[0, 1]
    ax.plot(epochs, logs['d_loss'], 'r-', linewidth=1.5, label='Train D Loss')
    ax.set_title('Discriminator Loss vs Epochs', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3. G Adversarial vs L1
    ax = axes[1, 0]
    ax.plot(epochs, logs['g_adv'], 'g-',  linewidth=1.5, label='G Adversarial')
    ax.plot(epochs, logs['g_l1'],  'm--', linewidth=1.5, label='G L1 (raw)')
    ax.set_title('Generator Components', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 4. Validation L1
    ax = axes[1, 1]
    ax.plot(epochs, logs['val_l1'], 'c-', linewidth=1.5, label='Val L1')
    ax.set_title('Validation L1 Loss vs Epochs', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('L1 Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Loss curves saved to: {save_path}")
    plt.show()

plot_training_logs(logs, save_path=os.path.join(cfg.OUTPUT_DIR, "loss_curves.png"))

In [ ]:
@torch.no_grad()
def evaluate_metrics(G, dataloader, device, max_batches=None):
    """
    Compute SSIM and PSNR on test set.

    Returns:
        dict with 'ssim_mean', 'ssim_std', 'psnr_mean', 'psnr_std'
    """
    G.eval()
    ssim_scores = []
    psnr_scores = []

    for batch_i, (sketch, real) in enumerate(dataloader):
        if max_batches and batch_i >= max_batches:
            break

        sketch = sketch.to(device)
        real   = real.to(device)

        with autocast(enabled=cfg.USE_AMP):
            fake = G(sketch)

        # Denormalize: [-1,1] → [0,1]
        fake_np = ((fake.cpu().numpy() * 0.5 + 0.5).clip(0, 1)
                   .transpose(0, 2, 3, 1))   # (B, H, W, C)
        real_np = ((real.cpu().numpy() * 0.5 + 0.5).clip(0, 1)
                   .transpose(0, 2, 3, 1))

        for f, r in zip(fake_np, real_np):
            # SSIM
            s = ssim(r, f, data_range=1.0, channel_axis=-1)
            # PSNR
            p = psnr(r, f, data_range=1.0)
            ssim_scores.append(s)
            psnr_scores.append(p)

    G.train()
    return {
        'ssim_mean': np.mean(ssim_scores),
        'ssim_std':  np.std(ssim_scores),
        'psnr_mean': np.mean(psnr_scores),
        'psnr_std':  np.std(psnr_scores),
        'n_samples': len(ssim_scores)
    }


print("Evaluating on test set...")
metrics = evaluate_metrics(G, test_loader, cfg.DEVICE)

print("\n" + "=" * 50)
print("QUANTITATIVE EVALUATION RESULTS")
print("=" * 50)
print(f"  Samples evaluated : {metrics['n_samples']}")
print(f"  SSIM  : {metrics['ssim_mean']:.4f} ± {metrics['ssim_std']:.4f}")
print(f"  PSNR  : {metrics['psnr_mean']:.2f} ± {metrics['psnr_std']:.2f} dB")
print("=" * 50)
print("\nInterpretation:")
print("  SSIM: 0=worst, 1=best   (>0.7 is good for sketch→photo)")
print("  PSNR: higher is better  (>25dB is acceptable for this task)")

In [ ]:
@torch.no_grad()
def final_visualization(G, dataset, device, n=10, save_path=None):
    """
    Final comparison grid showing n samples:
    Row 1: Input Sketch
    Row 2: Generated Photo
    Row 3: Ground Truth Photo
    """
    G.eval()

    n = min(n, len(dataset))
    indices = random.sample(range(len(dataset)), n)

    def denorm(t):
        return (t * 0.5 + 0.5).clamp(0, 1)

    fig = plt.figure(figsize=(2.8*n, 8.5))
    fig.suptitle(
        "Pix2Pix Results — CUHK Face Sketch\n"
        "Row 1: Input Sketch  |  Row 2: Generated Photo  |  Row 3: Ground Truth",
        fontsize=12, fontweight='bold'
    )

    gs = gridspec.GridSpec(3, n, figure=fig, hspace=0.08, wspace=0.05)

    row_titles = ['Input\nSketch', 'Generated\nPhoto', 'Ground\nTruth']
    row_colors = ['#2196F3', '#4CAF50', '#FF5722']

    for col, idx in enumerate(indices):
        sketch, real = dataset[idx]
        sketch_t = sketch.unsqueeze(0).to(device)
        fake     = G(sketch_t).cpu()

        images = [
            denorm(sketch).permute(1, 2, 0).numpy(),
            denorm(fake[0]).permute(1, 2, 0).numpy(),
            denorm(real).permute(1, 2, 0).numpy(),
        ]

        for row, img in enumerate(images):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(img)
            ax.axis('off')

            if col == 0:
                ax.text(-0.15, 0.5, row_titles[row],
                        transform=ax.transAxes,
                        fontsize=9, va='center', ha='right',
                        color=row_colors[row], fontweight='bold')

            if row == 0:
                ax.set_title(f"Sample {col+1}", fontsize=8)

    plt.subplots_adjust(left=0.12)

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Visualization saved: {save_path}")

    plt.show()
    G.train()


# Run final visualization on test set
final_visualization(
    G, test_dataset, cfg.DEVICE, n=10,
    save_path=os.path.join(cfg.OUTPUT_DIR, "final_results.png")
)

In [ ]:
print("\n" + "═"*60)
print("   FINAL EVALUATION SUMMARY")
print("═"*60)
print(f"  Model       : Pix2Pix (U-Net Generator + PatchGAN)")
print(f"  Dataset     : CUHK Face Sketch Database")
print(f"  Task        : Sketch → Realistic Face Photo")
print(f"  Image Size  : {cfg.IMG_SIZE}×{cfg.IMG_SIZE}")
print(f"  Epochs      : {cfg.NUM_EPOCHS}")
print(f"  Batch Size  : {cfg.BATCH_SIZE}")
print(f"  Lambda L1   : {cfg.LAMBDA_L1}")
print("─"*60)
print(f"  SSIM        : {metrics['ssim_mean']:.4f} ± {metrics['ssim_std']:.4f}")
print(f"  PSNR        : {metrics['psnr_mean']:.2f} ± {metrics['psnr_std']:.2f} dB")
print(f"  Test samples: {metrics['n_samples']}")
print("─"*60)
print(f"  Final Train G Loss : {logs['g_loss'][-1]:.4f}")
print(f"  Final Train D Loss : {logs['d_loss'][-1]:.4f}")
print(f"  Final Val   L1     : {logs['val_l1'][-1]:.4f}")
print("═"*60)

In [ ]:
import gradio as gr

# ── Load best model for inference ──────────────────────────────────────────
# Use the trained G directly (or load from checkpoint)
G_infer = G
G_infer.eval()

# Inference transform
infer_transform = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

def denorm_to_pil(tensor):
    """Convert model output tensor to PIL Image."""
    img = tensor.squeeze(0).cpu()
    img = (img * 0.5 + 0.5).clamp(0, 1)
    img = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(img)


@torch.no_grad()
def sketch_to_photo(sketch_pil):
    """
    Gradio inference function.
    Input : PIL Image (sketch / grayscale / edge)
    Output: PIL Image (generated realistic photo)
    """
    if sketch_pil is None:
        return None

    # Convert to RGB if grayscale
    sketch_pil = sketch_pil.convert('RGB')

    # Transform
    sketch_t = infer_transform(sketch_pil).unsqueeze(0).to(cfg.DEVICE)

    # Generate
    with autocast(enabled=cfg.USE_AMP):
        fake_t = G_infer(sketch_t)

    return denorm_to_pil(fake_t)


# ── Gradio Interface ────────────────────────────────────────────────────────
demo = gr.Interface(
    fn=sketch_to_photo,
    inputs=gr.Image(
        type="pil",
        label="Input: Face Sketch / Edge Image",
        image_mode="RGB"
    ),
    outputs=gr.Image(
        type="pil",
        label="Output: Generated Realistic Photo"
    ),
    title="🎨 Pix2Pix — Sketch to Realistic Face",
    description=(
        "Upload a face sketch or edge image and the model will generate "
        "a realistic face photo using the Pix2Pix Conditional GAN.\n\n"
        "**Model:** U-Net Generator + PatchGAN Discriminator  \n"
        "**Trained on:** CUHK Face Sketch Database"
    ),
    examples=[
        # Add paths to example sketches from the dataset
        [test_pairs[0][1]] if test_pairs else None,
        [test_pairs[1][1]] if len(test_pairs) > 1 else None,
        [test_pairs[2][1]] if len(test_pairs) > 2 else None,
    ],
    flagging_mode="never",
    theme=gr.themes.Soft()
)

print("Launching Gradio app...")
demo.launch(share=True, debug=False)